# interpkit Quickstart

This notebook walks through the core operations available in interpkit.
We'll load GPT-2 and run inspect, causal tracing, logit lens, attribution, and activation patching.

In [ ]:
import interpkit

model = interpkit.load("gpt2")

## Inspect

See the full module tree — types, parameter counts, shapes, and auto-detected roles (attention, MLP, norm, etc.).

In [ ]:
model.inspect()

## Logit Lens

Project each layer's hidden state into vocabulary space to see how the model's prediction evolves layer by layer.

In [ ]:
results = model.lens("The capital of France is")

## Causal Tracing

Which modules are most responsible for the model knowing that Paris is the capital of France?

We compare a clean prompt against a corrupted one and measure how much restoring each module's activation recovers the original answer.

In [ ]:
results = model.trace(
    "The capital of France is",
    "The capital of Poland is",
    top_k=15,
)

## Attribution

Gradient saliency — which input tokens matter most for the model's prediction?

In [ ]:
model.attribute("The capital of France is")

## Activation Patching

Swap a single module's output from the clean run into the corrupted run. A high effect means that module is critical for the factual recall.

In [ ]:
result = model.patch(
    "The capital of France is",
    "The capital of Poland is",
    at="transformer.h.8.mlp",
)
print(f"Patch effect: {result['effect']:.4f}")

## Ablation

Zero out a module's activations and measure how much the output changes.

In [ ]:
result = model.ablate(
    "The capital of France is",
    at="transformer.h.8.mlp",
    method="zero",
)
print(f"Ablation effect: {result['effect']:.4f}")

## Raw Activations

Extract activation tensors at any named module for your own analysis.

In [ ]:
act = model.activations("The capital of France is", at="transformer.h.8.mlp")
print(f"Shape: {act.shape}")
print(f"Norm:  {act.float().norm():.2f}")